In [3]:
import math
import os
import hydra
import tonic
import torch
from tonic.transforms import Optional, ToFrame
from datasets.utils.pad_tensors import PadTensors
from datasets.utils.diskcache import DiskCachedDataset
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
import numpy as np

/home/michael/projects/SE-adlif/.venv/lib/python3.11/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Data Preparation
We first need to preprocess the raw data found in the data path into a **Tonic** dataset. 

In [23]:
data_path = "/raid/home/michael.siegl/datasets/welding-data/raw"
# identify files, ending either in .raw or .bias
raw_files = []
bias_files = []
# check path
print(f"Checking path: {data_path}")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Path {data_path} does not exist.")

for file in os.listdir(data_path):
    if file.endswith(".raw"):
        raw_files.append(file)
    elif file.endswith(".bias"):
        bias_files.append(file)

# sort files by name, to ensure matching raw and bias files
raw_files.sort()
bias_files.sort()

Checking path: /raid/home/michael.siegl/datasets/welding-data/raw


In [15]:
# inspect a bias file
bias_file = bias_files[0]
print(f"Inspecting bias file: {bias_file}")
with open(os.path.join(data_path, bias_file), "r") as f:
    bias_data = f.read()
print(f"Bias data:\n{bias_data}")

Inspecting bias file: recording10.bias
Bias data:
51   % bias_diff
28   % bias_diff_off
25   % bias_diff_on
34   % bias_fo
40   % bias_hpf
10   % bias_refr



In [19]:
raw_files

['recording03.raw',
 'recording04.raw',
 'recording05.raw',
 'recording06.raw',
 'recording07.raw',
 'recording08.raw',
 'recording09.raw',
 'recording10.raw',
 'recording11.raw',
 'recording12.raw',
 'recording13.raw',
 'recording14.raw',
 'recording15.raw',
 'recording16.raw',
 'recording17.raw',
 'recording18.raw',
 'recording19.raw',
 'recording20-120cmpm.raw',
 'recording20-30cmpm.raw',
 'recording20-60cmpm.raw',
 'recording20-moving1.raw',
 'recording20-moving2.raw',
 'recording20-moving3.raw',
 'recording20-moving4.raw',
 'recording20-moving5.raw',
 'recording20-moving6.raw',
 'recording20-welding-120cmpm.raw',
 'recording20-welding-30cmpm.raw',
 'recording20-welding-60cmpm.raw']

In [24]:
# drop files 13,14,15 as they are corrupted
raw_files = raw_files[:11] + raw_files[14:]
bias_files = bias_files[:13] + bias_files[16:]
print(f"Remaining raw files: {raw_files}")

Remaining raw files: ['recording03.raw', 'recording04.raw', 'recording05.raw', 'recording06.raw', 'recording07.raw', 'recording08.raw', 'recording09.raw', 'recording10.raw', 'recording11.raw', 'recording12.raw', 'recording13.raw', 'recording17.raw', 'recording18.raw', 'recording19.raw', 'recording20-120cmpm.raw', 'recording20-30cmpm.raw', 'recording20-60cmpm.raw', 'recording20-moving1.raw', 'recording20-moving2.raw', 'recording20-moving3.raw', 'recording20-moving4.raw', 'recording20-moving5.raw', 'recording20-moving6.raw', 'recording20-welding-120cmpm.raw', 'recording20-welding-30cmpm.raw', 'recording20-welding-60cmpm.raw']


## Statistics
I should probably take some statistics over this dataset, like total event counts across the videos, jitter/latency

In [25]:
# okay for now I will just go with the recording20 files with speeds 30cmpm, 60cmpm, 120cmpm
speed_files = [f for f in raw_files if "recording20" in f and any(speed in f for speed in ["30cmpm", "60cmpm", "120cmpm"])]
print(f"Selected speed files: {speed_files}")

Selected speed files: ['recording20-120cmpm.raw', 'recording20-30cmpm.raw', 'recording20-60cmpm.raw', 'recording20-welding-120cmpm.raw', 'recording20-welding-30cmpm.raw', 'recording20-welding-60cmpm.raw']


In [26]:
# create folders for the 3 speeds to use the torch Dataset folder for loading and put the raw files in the folders
for speed in ["30cmpm", "60cmpm", "120cmpm"]:
    folder_path = os.path.join(data_path, speed)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    for file in speed_files:
        if speed in file:
            os.rename(os.path.join(data_path, file), os.path.join(folder_path, file))

In [43]:
speed_files

['recording20-120cmpm.raw',
 'recording20-30cmpm.raw',
 'recording20-60cmpm.raw',
 'recording20-welding-120cmpm.raw',
 'recording20-welding-30cmpm.raw',
 'recording20-welding-60cmpm.raw']

In [47]:
file_path = os.path.join(data_path, "30cmpm", speed_files[1])
print(f"Inspecting file: {file_path}")
with open(file_path, "rb") as f:
    raw_data = f.read()
print(f"Raw data length: {len(raw_data)} bytes")
print(raw_data[:500])  # print first 100 bytes to inspect the format    

Inspecting file: /raid/home/michael.siegl/datasets/welding-data/raw/30cmpm/recording20-30cmpm.raw
Raw data length: 1465677224 bytes
b"% camera_integrator_name rp1-cfe\n% date 2026-02-16 19:34:36\n% format EVT21;height=320;width=320\n% generation 320.0\n% geometry 320x320\n% plugin_integrator_name Prophesee\n% plugin_name hal_plugin_prophesee\n% sensor_name GenX320\n% serial_number genx320 10-003c\n% end\n\x00\x00\x00\x00\xfcW\x08\x80\x00\x08\x00\x00\x8b\x00\x05\x18\x80\x00\x00\x00G\x00A\x08\x00\x00\x00@\x0e\x00C\x18\x00\x00@\x00\x15\x01\x88\x08\x00\x00\x00 ;\x01\xc1\x18\x02\x00\x00\x00\xc2\x00D\t\x01\x00\x00\x00\x91\x00\x82\x19\x00\x00\x08\x00\xdc\x00\xc3\t\x08\x00\x00\x00\x16\x01\xc3\t\x00\x00\x10\x00\xe8\x00\x00\n\x00@\x00\x00\x17\x01H\n\x00\x00\x01\x00\t\x01\x88\n\x00\x00\x01\x00/\x01B\x1b\x00\x10\x00\x00\xfd\x00E\x0b\x00\x00\x01\x00\x11\x00@\x1b\x01\x00\x00\x00\xf7\x00\xc3\x0b\x00\x00\x00\x00\xfcW\x08\x80\x00\x00\x80\x00_\x00H\x1c\x00\x00\x00\x04\xce\x00B\x0c\x00\x00\x02\x00\xa6\x0

Okay, great, now we have a 3 class dataset with 2 recordings per speed. We do however still need to remove the parts where there is no actual movement, I will do this by doing event based binning ... hmm yeah we'll see

In [15]:
# the used sensor is the GenX320 Metavision sensor with sensor size 320x320
sensor_size = (320, 320, 2)

In [11]:
from pandas import DataFrame, read_csv
csv_path = "./recording1.csv"
df = read_csv(csv_path, header=None, names=["x", "y", "p", "t"])
event_data = df.to_numpy()
print(f"Event data shape: {event_data.shape}")

Event data shape: (12701989, 4)


In [16]:
# great now we're gonna use the to frame transformer to convert the raw data into frames and visualize them
_event_to_tensor = ToFrame(sensor_size=sensor_size, n_time_bins=50)
frames = _event_to_tensor(event_data)
print(f"Frames shape: {frames.shape}")

TypeError: argument of type 'NoneType' is not iterable

In [38]:
class MyRecordings(tonic.Dataset):
    sensor_size = (
        320,
        320,
        2,
    )  # the sensor size of the event camera or the number of channels of the silicon cochlear that was used
    ordering = (
        "xytp"  # the order in which your event channels are provided in your recordings
    )

    def __init__(
        self,
        train=True,
        transform=None,
        target_transform=None,
    ):
        super(MyRecordings, self).__init__(
            save_to='./', transform=transform, target_transform=target_transform
        )
        
        classes = ["30cmpm", "60cmpm", "120cmpm"]
        self.filenames = []
        for cls in classes:
            cls_folder = os.path.join(data_path, cls)
            for file in os.listdir(cls_folder):
                if file.endswith(".raw"):
                    self.filenames.append(os.path.join(cls_folder, file))

    def __getitem__(self, index):
        events = np.load(self.filenames[index], allow_pickle=True)

        if self.transform is not None:
            events = self.transform(events)

        return events

    def __len__(self):
        return len(self.filenames)

In [ ]:
class WeldingSpeed(tonic.datasets.Dataset):
    def __init__(self, root=data_path, transform=None, target_transform=None):
        super().__init__(root, transform, target_transform)

        self.classes = ["speed1", "speed2", "speed3"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        events = sample["events"]
        label = sample["label"]
        if self.transform:
            events = self.transform(events)
        if self.target_transform:
            label = self.target_transform(label)
        return events, label